# Alternativas a OpenAI — Ollama y Gemini

_Modelos open-source corriendo en tu máquina y la API de Google_

**Módulo 3 — Introducción a AI Engineering** | DSRP Machine Learning Engineering
**Profesor:** Miguel Arquez

![AI Engineering](assets/header.png)

## 1. ¿Por qué considerar alternativas a OpenAI?

OpenAI es el _default_ pero rara vez es la única opción razonable. Razones para mirar alternativas:

| Razón | Explicación |
|---|---|
| **Privacidad / regulación** | Tus datos (médicos, legales, financieros) no pueden salir de tu infraestructura. Necesitas modelos locales. |
| **Costo** | Volumen alto + tarea simple → un modelo open-source self-hosted o un Haiku/Flash sale 10-100× más barato. |
| **Latencia** | Llamada local = sub-100ms. Inferencia en hardware especializado (Groq, Cerebras) = 5-10× más rápida que GPU típica. |
| **Vendor lock-in** | Diversificar proveedores te protege de cambios de pricing, deprecaciones de modelos, downtime. |
| **Customización** | Open-source te deja fine-tunear, cuantizar, modificar la arquitectura. |
| **Frontera** | A veces _no_ es OpenAI quien tiene el mejor modelo para tu tarea (Claude para código, Gemini para multimodal de contexto largo). |

En este notebook cubrimos tres alternativas representativas:
- **Ollama** — runtime local para modelos open-source (Llama, Mistral, Qwen, Gemma, …).
- **Gemini** — la familia de modelos de Google, con context window enorme.
- **Groq** — inferencia ultra-rápida de modelos open-source sobre hardware especializado (LPU).

## 2. Cargar credenciales desde `ai.env`

Igual que en el notebook 02, usamos un archivo `ai.env` (ya está en `.gitignore`) junto al notebook con todas las keys que vamos a usar:

```
# modulo-3-introduccion-ai-engineering/ai.env
OPENAI_API_KEY=sk-...        # opcional aquí (truco "Ollama con SDK de OpenAI")
GOOGLE_API_KEY=...           # https://aistudio.google.com/app/apikey
GROQ_API_KEY=gsk_...         # https://console.groq.com/keys
HF_TOKEN=hf_...              # opcional, no se usa en este notebook
```

Cada proveedor te da su key en su propia consola. Ninguno necesita tarjeta para el tier gratuito.

In [ ]:
# Cargar credenciales desde ai.env (mismo patrón que el notebook 02)
import os
from pathlib import Path
from dotenv import load_dotenv

env_path = Path('ai.env') if Path('ai.env').exists() else Path('.env')
load_dotenv(env_path)

def _mask(v):
    return f'{v[:6]}…{v[-4:]}' if v and len(v) > 12 else '***'

for k in ('OPENAI_API_KEY', 'GOOGLE_API_KEY', 'GROQ_API_KEY', 'HF_TOKEN'):
    v = os.environ.get(k)
    flag = '✅' if v else 'ℹ️ '
    print(f'{flag} {k:<16} ({_mask(v) if v else "no configurada"})')

print(f'\nFuente: {env_path}')

## 3. El ecosistema de modelos open-source

Una muestra de los modelos open-source más usados en 2025-2026:

| Modelo | Tamaños | Fuerte en | Licencia |
|---|---|---|---|
| **Llama 3.1 / 3.3** (Meta) | 8B / 70B / 405B | propósito general, instrucciones | Llama Community License |
| **Mistral / Mixtral** | 7B / 8x7B / 8x22B | eficiencia, MoE | Apache 2.0 (algunos) |
| **Qwen 2.5** (Alibaba) | 0.5B – 72B | multilingüe, código | Apache 2.0 |
| **Gemma 2** (Google) | 2B / 9B / 27B | tamaños pequeños eficientes | Gemma License |
| **Phi-3 / Phi-4** (Microsoft) | 3.8B – 14B | razonamiento por su tamaño | MIT |
| **DeepSeek V3 / R1** | 67B+ | razonamiento (R1), código (V3) | MIT |

> "Open-source" en LLMs es un espectro: algunos liberan **pesos + código + datos**, la mayoría solo **pesos** con licencia que limita uso comercial.

## 4. Ollama — qué es y cómo funciona

[Ollama](https://ollama.com/) es un **runtime local** para LLMs open-source. Empaqueta el modelo, los pesos cuantizados y un servidor HTTP en una sola herramienta. Comparable a "Docker para LLMs".

**Características clave:**
- Un comando para descargar y correr cualquier modelo: `ollama run llama3.2`.
- API HTTP local (`http://localhost:11434`) **compatible con OpenAI** — puedes apuntar el SDK de OpenAI a Ollama cambiando solo `base_url`.
- Cuantización por defecto (4-bit) — un modelo de 8B corre cómodamente en 8 GB de RAM.
- Mac, Linux, Windows. Aprovecha GPU si está disponible (Metal en Mac, CUDA en NVIDIA).

### 4.1 Instalación

**macOS (con Homebrew):**
```bash
brew install ollama
```

**Linux:**
```bash
curl -fsSL https://ollama.com/install.sh | sh
```

**Windows:** descarga el instalador desde https://ollama.com/download

### 4.2 Descargar y correr un modelo

```bash
# Descargar el modelo (la primera vez baja ~5 GB para Llama 3.2 8B)
ollama pull llama3.2

# Modo chat interactivo en la terminal
ollama run llama3.2

# El servidor HTTP corre en background al instalar; si no:
ollama serve
```

Lista de modelos disponibles: https://ollama.com/library  (llama3.2, mistral, qwen2.5, gemma2, phi3, deepseek-r1, codellama, …)

### 4.3 Python client

```bash
uv add ollama
```

In [ ]:
# Chat con Ollama — requiere `ollama serve` corriendo en localhost:11434
# uv add ollama
import ollama

resp = ollama.chat(
    model='llama3.2',
    messages=[
        {'role': 'system', 'content': 'Eres un asistente conciso. Responde en máximo 3 frases.'},
        {'role': 'user',   'content': '¿Qué es un Transformer?'},
    ],
    options={
        'temperature': 0.3,
        'num_predict': 150,        # equivalente a max_tokens
    },
)
print(resp['message']['content'])
print('---')
print(f"Tokens evaluados: {resp.get('eval_count', 'n/a')}")

In [ ]:
# Streaming token a token
import ollama

stream = ollama.chat(
    model='llama3.2',
    messages=[{'role': 'user', 'content': 'Cuenta del 1 al 5 con una palabra creativa por número.'}],
    stream=True,
)

for chunk in stream:
    print(chunk['message']['content'], end='', flush=True)

In [ ]:
# Embeddings con Ollama (cero costo, todo local)
import ollama

textos = [
    'Me encanta programar en Python.',
    'Adoro escribir código en Python.',
    'El pan está caliente y crujiente.',
]

embs = []
for t in textos:
    r = ollama.embeddings(model='nomic-embed-text', prompt=t)
    embs.append(r['embedding'])

import numpy as np
embs = np.array(embs)
print('Shape:', embs.shape)

from sklearn.metrics.pairwise import cosine_similarity
print('\nSimilitud coseno:')
print(cosine_similarity(embs).round(2))

### 4.4 Truco: usar el SDK de OpenAI apuntando a Ollama

Como Ollama expone una API compatible, puedes reusar tu código de OpenAI cambiando solo dos parámetros. Ideal para alternar entre cloud y local sin tocar el resto del código:

```python
from openai import OpenAI

# Apuntamos al servidor local de Ollama
client = OpenAI(
    base_url='http://localhost:11434/v1',
    api_key='ollama',          # cualquier string vale, no la valida
)

resp = client.chat.completions.create(
    model='llama3.2',
    messages=[{'role': 'user', 'content': 'Hola, ¿quién eres?'}],
)
print(resp.choices[0].message.content)
```

Esto te da una **abstracción de proveedor gratis**: cambias `base_url` y vas de cloud a local. Es el mismo truco que usaremos en un momento con Groq.

## 5. Gemini — la familia de modelos de Google

[Gemini](https://ai.google.dev/) es el LLM frontera de Google. Familias principales:

| Modelo | Fuerte en | Context window |
|---|---|---|
| **Gemini 2.5 Pro** | razonamiento profundo, código, multimodal | 2M tokens |
| **Gemini 2.5 Flash** | latencia y costo bajos, alta calidad | 1M tokens |
| **Gemini 2.5 Flash-Lite** | volumen masivo a costo mínimo | 1M tokens |
| **Gemma 2 / 3** | open weights (corre en Ollama) | varía |

**Diferenciadores frente a OpenAI:**
- **Context window enorme** (1-2 M tokens) — útil para mandar PDFs gigantes, libros, repos enteros.
- **Multimodal nativo** — texto + imagen + video + audio en la misma llamada.
- **Pricing competitivo** — Flash es de los más baratos del mercado.
- **Tier gratuito generoso** para prototipado vía Google AI Studio.

### 5.1 Setup

```bash
# 1. Crear key gratis en https://aistudio.google.com/app/apikey
# 2. Agregar a ai.env:  GOOGLE_API_KEY=...
# 3. Instalar el SDK
uv add google-generativeai
```

La key ya quedó cargada en la sección 2 desde `ai.env` — solo la usamos abajo.

In [ ]:
# Chat con Gemini
# uv add google-generativeai
import os
import google.generativeai as genai

genai.configure(api_key=os.environ.get('GOOGLE_API_KEY'))

model = genai.GenerativeModel(
    model_name='gemini-2.5-flash',
    system_instruction='Eres un asistente conciso. Responde en máximo 3 frases.',
)

resp = model.generate_content(
    '¿Qué diferencia hay entre RNN y Transformer?',
    generation_config={
        'temperature': 0.3,
        'max_output_tokens': 200,
        'top_p': 0.9,
        'top_k': 40,
    },
)
print(resp.text)

In [ ]:
# Conversación multi-turno con Gemini
chat = model.start_chat(history=[])
print('USER:', '¿Qué es un embedding?')
print('BOT :', chat.send_message('¿Qué es un embedding?').text)
print('---')
print('USER:', 'Dame un caso de uso concreto.')
print('BOT :', chat.send_message('Dame un caso de uso concreto.').text)
print('---')
print(f'Mensajes en el historial: {len(chat.history)}')

In [ ]:
# Embeddings con Gemini
import google.generativeai as genai

textos = [
    'Me encanta programar en Python.',
    'Adoro escribir código en Python.',
    'El pan está caliente y crujiente.',
    'La pizza italiana es deliciosa.',
]

resp = genai.embed_content(
    model='models/text-embedding-004',
    content=textos,
    task_type='semantic_similarity',
)

import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
embs = np.array(resp['embedding'])
print('Shape:', embs.shape)
print('\nSimilitud coseno:')
print(cosine_similarity(embs).round(2))

In [ ]:
# Multimodal nativo: pasarle una imagen + texto
import PIL.Image
import requests
from io import BytesIO

# Bajamos una imagen pública
url = 'https://upload.wikimedia.org/wikipedia/commons/thumb/4/47/PNG_transparency_demonstration_1.png/280px-PNG_transparency_demonstration_1.png'
img = PIL.Image.open(BytesIO(requests.get(url).content))

resp = model.generate_content([
    'Describe esta imagen en una sola frase.',
    img,
])
print(resp.text)

### 5.2 Streaming con Gemini

```python
stream = model.generate_content('Cuenta una historia corta', stream=True)
for chunk in stream:
    if chunk.text:
        print(chunk.text, end='', flush=True)
```

### 5.3 Function calling

Gemini también soporta tool use con una API parecida a OpenAI:

```python
def get_weather(city: str) -> dict:
    return {'city': city, 'temp_c': 22, 'desc': 'soleado'}

model = genai.GenerativeModel(
    'gemini-2.5-flash',
    tools=[get_weather],   # SDK envuelve la firma automáticamente
)
chat = model.start_chat(enable_automatic_function_calling=True)
print(chat.send_message('¿Qué tiempo hace en Lima?').text)
```

## 6. Groq — inferencia ultra-rápida con LPUs

[Groq](https://groq.com/) sirve modelos open-source corriendo sobre hardware **LPU (Language Processing Unit)** — chips diseñados específicamente para inferencia de LLMs. Resultado: latencias y throughput **muy superiores** a GPUs convencionales.

| | Groq (LPU) | GPU típica en cloud |
|---|---|---|
| Throughput | **300-800 tokens/s** | 30-100 tokens/s |
| Latencia primer token | ~50 ms | 200-500 ms |
| Modelos | Llama 3.1/3.3, Mixtral, Gemma, Qwen, DeepSeek R1 | depende del proveedor |

**Por qué importa:** en aplicaciones interactivas (chat, agentes, voz), la velocidad cambia la sensación del producto. Una respuesta que tarda 500 ms vs 5 s es la diferencia entre "instantáneo" y "esperando".

**Otras ventajas:**
- **Tier gratuito generoso** — 30 RPM en la mayoría de modelos, sin tarjeta.
- **API compatible con OpenAI** — solo cambias `base_url` y `api_key`. Mismo truco que con Ollama.
- **Modelos open-source** — no estás casado con un modelo propietario; si Groq mañana sube precios, te llevas el código y lo corres en otro proveedor con los mismos pesos.

### 6.1 Setup

```bash
# 1. Crear key gratis en https://console.groq.com/keys
# 2. Agregar a ai.env:  GROQ_API_KEY=gsk_...
# 3. El SDK de OpenAI ya está instalado, no hace falta uno nuevo.
```

La key ya quedó cargada en la sección 2 desde `ai.env`.

In [ ]:
# Groq usando el SDK de OpenAI — el truco está en base_url
import os, time
from openai import OpenAI

groq = OpenAI(
    base_url='https://api.groq.com/openai/v1',
    api_key=os.environ.get('GROQ_API_KEY'),
)

t0 = time.time()
resp = groq.chat.completions.create(
    model='llama-3.3-70b-versatile',
    messages=[
        {'role': 'system', 'content': 'Eres un asistente conciso. Responde en español, máximo 3 frases.'},
        {'role': 'user',   'content': 'Resume qué es Groq y por qué importa.'},
    ],
    temperature=0.3,
    max_tokens=150,
)
elapsed = time.time() - t0

print(resp.choices[0].message.content)
print('---')
print(f'Tokens output: {resp.usage.completion_tokens}')
print(f'Tiempo total : {elapsed:.2f}s  →  {resp.usage.completion_tokens/elapsed:.0f} tok/s')

### 6.2 Streaming con Groq

Aquí es donde se nota la velocidad — el texto aparece tan rápido que apenas alcanzas a leerlo. Útil para experiencias tipo "instantáneo".

In [ ]:
# Streaming con Groq — fíjate en la velocidad de tokens/segundo
import time

t0 = time.time()
first_token_at = None
n_tokens = 0

stream = groq.chat.completions.create(
    model='llama-3.3-70b-versatile',
    messages=[{'role': 'user', 'content': 'Cuenta del 1 al 10 con una palabra creativa por número.'}],
    stream=True,
)

for chunk in stream:
    delta = chunk.choices[0].delta.content
    if delta:
        if first_token_at is None:
            first_token_at = time.time() - t0
        print(delta, end='', flush=True)
        n_tokens += 1

elapsed = time.time() - t0
print(f'\n\n---')
print(f'Time to first token : {first_token_at*1000:.0f} ms')
print(f'Tiempo total        : {elapsed:.2f} s')
print(f'Throughput          : {n_tokens/elapsed:.0f} tok/s (aproximado)')

### 6.3 Modelos disponibles en Groq

| Modelo | Tokens / s aprox. | Notas |
|---|---|---|
| `llama-3.3-70b-versatile` | ~280 | propósito general, calidad alta |
| `llama-3.1-8b-instant` | ~750 | muy rápido para tareas simples |
| `mixtral-8x7b-32768` | ~500 | MoE, contexto 32K |
| `gemma2-9b-it` | ~500 | versión de Gemma instruction-tuned |
| `deepseek-r1-distill-llama-70b` | ~250 | reasoning, devuelve `<think>...</think>` |
| `qwen-2.5-32b` | ~200 | multilingüe, código |

Catálogo completo y precios: https://console.groq.com/docs/models

> 💡 **Cuándo usar Groq**: aplicaciones donde la latencia importa (chat en tiempo real, voz, agentes con muchos pasos), prototipado sin gastar créditos de OpenAI, fallback de un proveedor caído. **Cuándo NO**: cuando necesitas un modelo propietario específico (GPT-4o, Claude Opus) — Groq solo sirve open-source.

## 7. Comparación entre proveedores

| | OpenAI | Anthropic | Gemini | **Groq** | Ollama (local) |
|---|---|---|---|---|---|
| **Modelos** | GPT-4o, GPT-5, o3 | Claude Opus / Sonnet / Haiku 4.x | Gemini 2.5 Pro / Flash | Llama, Mixtral, Qwen, R1 (open) | Llama, Mistral, Qwen, Gemma, … |
| **Pricing** | medio – alto | medio – alto | bajo – medio | **bajo + tier gratis generoso** | gratis (HW propio) |
| **Multimodal** | ✅ (vision, audio, TTS) | ✅ (vision, audio) | ✅✅ nativo (incluye video) | parcial (vision en algunos) | limitado (algunos VLMs) |
| **Context window** | 128K – 1M | 200K – 1M | 1M – 2M | 32K – 128K | 8K – 128K (depende del modelo) |
| **Razonamiento** | o1, o3 | thinking en Opus 4 | thinking en 2.5 Pro | DeepSeek R1 | DeepSeek R1, QwQ |
| **Privacidad** | cloud | cloud | cloud | cloud | **local total** |
| **Latencia** | red + inferencia | red + inferencia | red + inferencia | **red + inferencia ultra-rápida (LPU)** | solo inferencia local |
| **Lock-in** | alto | alto | alto | bajo (modelos open-source) | ninguno |
| **API compatible OpenAI SDK** | ✅ (nativa) | ❌ | parcial | ✅ (`base_url` swap) | ✅ (`base_url` swap) |
| **Mejor para** | default seguro, ecosistema | código y escritura | contexto largo + multimodal | **velocidad bestial + costo bajo** | privacidad, prototipado offline, costo cero |

## 8. Cuándo elegir cada uno

- **Default razonable para arrancar** → GPT-4o-mini o Claude Sonnet 4. Buena calidad/precio, ecosistema maduro.
- **Tarea frontera (código difícil, razonamiento, escritura)** → Claude Opus 4.x con extended thinking, GPT-5, Gemini 2.5 Pro thinking.
- **Volumen masivo, tarea simple (clasificación, extracción)** → Haiku 4, GPT-4o-mini, Gemini 2.5 Flash-Lite o `llama-3.1-8b-instant` en Groq.
- **Documentos largos / repos enteros** → Gemini 2.5 (1-2M de contexto) o Claude (200K + caching).
- **Latencia mínima (chat tiempo real, voz, agentes multi-paso)** → **Groq** (300-800 t/s, time-to-first-token ~50 ms).
- **Probar modelos open-source sin instalar nada** → **Groq** te corre Llama 3.3 70B desde una API en segundos, con tier gratuito.
- **Datos sensibles que no pueden salir de tu infra** → Ollama con Llama 3.x o Qwen 2.5 70B; o vLLM/TGI en tu cluster.
- **Sin internet (demos, talleres, prototipo offline)** → Ollama local.
- **Multimodal serio (video, audio largo)** → Gemini 2.5 Pro.

## 9. Patrón de abstracción del proveedor

En producción, **no acoples tu lógica a un SDK concreto**. Encapsula las llamadas detrás de una pequeña interfaz para poder cambiar de proveedor sin tocar el resto del código:

```python
class LLMClient:
    def chat(self, messages, **opts) -> str: ...

class OpenAIClient(LLMClient):
    def chat(self, messages, **opts):
        return openai.chat.completions.create(model='gpt-4o-mini',
                                              messages=messages, **opts).choices[0].message.content

class GroqClient(LLMClient):
    def __init__(self):
        self.client = OpenAI(base_url='https://api.groq.com/openai/v1',
                             api_key=os.environ['GROQ_API_KEY'])
    def chat(self, messages, **opts):
        return self.client.chat.completions.create(model='llama-3.3-70b-versatile',
                                                   messages=messages, **opts).choices[0].message.content

class GeminiClient(LLMClient): ...
class OllamaClient(LLMClient): ...

# El resto de tu app trabaja contra LLMClient, no contra openai/gemini/groq/etc.
```

Beneficios:
- **A/B testing** de proveedores por feature flag.
- **Fallback** automático si un proveedor cae (ej. si OpenAI 429, te vas a Groq).
- **Tests** con un fake `LLMClient` que devuelve respuestas determinísticas.
- **Migración** entre proveedores sin reescribir lógica de negocio.

> 🎯 Nota: como **Groq y Ollama** ya exponen API compatible OpenAI, su `chat()` es básicamente idéntico al de OpenAI — la abstracción te sale casi gratis.

## 10. Cierre del módulo

Lo que tienes ahora:

- Una **base conceptual** de qué son los LLMs y cómo aprendieron lo que saben (notebook 01).
- Un **mapa práctico** del ecosistema de APIs y cómo facturan (notebook 02).
- Las **palancas finas** de prompt + context engineering y entender lo que pasa por debajo (notebook 03).
- **Salidas alternativas** cuando OpenAI no es la mejor opción — Ollama, Gemini, Groq (este notebook).

A partir de aquí los siguientes pasos naturales son:
- **RAG** — combinar embeddings + un vector store + chat completions.
- **Agentes** — sistemas que usan tools y se planifican en varios pasos.
- **Evals** — cómo medir objetivamente que un cambio mejora.
- **Fine-tuning** — cuando el prompting ya no alcanza.

## 11. Referencias

**Ollama**
- Ollama — https://ollama.com/
- Ollama Python client — https://github.com/ollama/ollama-python
- Ollama OpenAI compatibility — https://ollama.com/blog/openai-compatibility

**Gemini**
- Google AI Studio (donde se generan keys de Gemini) — https://aistudio.google.com/
- Gemini API docs — https://ai.google.dev/gemini-api/docs
- google-generativeai SDK — https://github.com/google/generative-ai-python

**Groq**
- Groq — https://groq.com/
- Groq console (API keys) — https://console.groq.com/keys
- Groq model catalog & pricing — https://console.groq.com/docs/models
- Groq + OpenAI SDK compatibility — https://console.groq.com/docs/openai

**Ecosistema general**
- Hugging Face — Open LLM Leaderboard: https://huggingface.co/spaces/open-llm-leaderboard/open_llm_leaderboard
- Artificial Analysis — comparativa de modelos y pricing en tiempo real: https://artificialanalysis.ai/
- LMSYS Chatbot Arena (ranking por preferencias humanas) — https://lmarena.ai/